# Stage 2 — Activation patching and the ablation harness

**Capstone: Evaluating the Faithfulness of Attribution-Based Interpretability Tooling in GPT-2**

This is the methodological core. It builds the two tools every later experiment depends on:

1. **Activation patching** — the trusted way to measure how *important* each attention head is, by intervention.
2. **An ablation harness** — a simple function to switch heads off and measure the effect. This is how we will test *faithfulness* later.

**Plain-English idea.** Activation patching is like swapping one part from a working machine into a broken one to see how much it fixes things: if swapping in head X's activity rescues the behaviour, head X was important. Ablation is the opposite — take a working machine and switch a part off to see whether it breaks.

**How to run:** open in Google Colab, ideally with a GPU (Runtime → Change runtime type → T4 GPU), then Run all. It works on CPU but is slower.

*Tools used: TransformerLens (Nanda & Bloom, 2022). The patching approach follows Wang et al. (2023).*

## 1. Setup (same as Stage 1)

In [ ]:
!pip -q install transformer_lens

In [ ]:
import torch, functools
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from transformer_lens import HookedTransformer, utils

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)
n_layers, n_heads = model.cfg.n_layers, model.cfg.n_heads
print(f"GPT-2 small on {device}: {n_layers} layers x {n_heads} heads")

## 2. Build clean and corrupted sentence pairs
For patching we need two versions of each sentence:

- **Clean:** `When Mary and John went to the shop, John gave a drink to` → correct answer **Mary**.
- **Corrupted:** identical except the second name is swapped — `... Mary gave a drink to` → the correct answer flips to **John**.

The two sentences differ by a **single token**, which makes the comparison clean. Throughout, we always measure the same quantity: `logit(Mary) − logit(John)`. It is **positive** on the clean sentence and **negative** on the corrupted one. We use one sentence template so every sentence is the same length (this keeps the code simple).

In [ ]:
template = "When{A} and{B} went to the shop,{S} gave a drink to"
candidate_names = [" Mary", " John", " Tom", " James", " Anna", " Kate",
                   " Mark", " Paul", " Alice", " Sarah", " David", " Emma"]
names = [n for n in candidate_names if model.to_tokens(n, prepend_bos=False).shape[1] == 1]

import itertools, random
random.seed(0)
pairs = [(a, b) for a, b in itertools.permutations(names, 2)]
random.shuffle(pairs)
pairs = pairs[:15]   # 15 sentence pairs; increase later for smoother numbers

clean_prompts     = [template.format(A=a, B=b, S=b) for a, b in pairs]  # subject = B -> answer A
corrupted_prompts = [template.format(A=a, B=b, S=a) for a, b in pairs]  # subject = A -> answer flips
io_tokens = torch.tensor([model.to_single_token(a) for a, b in pairs], device=device)  # correct name (A)
s_tokens  = torch.tensor([model.to_single_token(b) for a, b in pairs], device=device)  # other name  (B)

clean_tokens     = model.to_tokens(clean_prompts)
corrupted_tokens = model.to_tokens(corrupted_prompts)
print("Example clean:    ", repr(clean_prompts[0]))
print("Example corrupted:", repr(corrupted_prompts[0]))
print("Token tensor shape:", tuple(clean_tokens.shape))

## 3. The metric and the two baselines
`logit_diff` returns the average of `logit(correct) − logit(other)` across all sentences. We expect the clean baseline to be clearly positive and the corrupted baseline to be negative or near zero.

In [ ]:
N = len(pairs)
rows = torch.arange(N, device=device)

def logit_diff(logits):
    final = logits[:, -1, :]                 # prediction for the next token, per sentence
    return (final[rows, io_tokens] - final[rows, s_tokens]).mean().item()

clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corrupted_logits = model(corrupted_tokens)

CLEAN_LD = logit_diff(clean_logits)
CORRUPT_LD = logit_diff(corrupted_logits)
print(f"Clean logit difference:     {CLEAN_LD:+.3f}   (should be clearly positive)")
print(f"Corrupted logit difference: {CORRUPT_LD:+.3f}   (should be negative / near zero)")

## 4. Activation patching: which heads matter?
For every attention head we do the following:

1. Run the **corrupted** sentence.
2. But **patch in** that one head's activity from the **clean** run.
3. Measure how far this moves the logit difference back towards the clean value.

We report a **restoration score** between 0 and 1:

`(patched − corrupted) / (clean − corrupted)`

A score near **1** means that single head almost fully restores the correct behaviour — it is important. Near **0** means it does nothing. `hook_z` is the head's output; we replace head `h`'s slice with the clean version.

In [ ]:
def patch_head_hook(z, hook, head, clean_cache):
    # z has shape [batch, position, head, d_head]; overwrite one head with its clean activity
    z[:, :, head, :] = clean_cache[hook.name][:, :, head, :]
    return z

denom = (CLEAN_LD - CORRUPT_LD)
importance = np.zeros((n_layers, n_heads))

for layer in range(n_layers):
    name = utils.get_act_name("z", layer)
    for head in range(n_heads):
        hook_fn = functools.partial(patch_head_hook, head=head, clean_cache=clean_cache)
        patched_logits = model.run_with_hooks(corrupted_tokens, fwd_hooks=[(name, hook_fn)])
        importance[layer, head] = (logit_diff(patched_logits) - CORRUPT_LD) / denom

print("Done. Patched every head (", n_layers * n_heads, "in total ).")

In [ ]:
# Heatmap of head importance (layer x head)
plt.figure(figsize=(8, 6))
plt.imshow(importance, cmap="RdBu", vmin=-abs(importance).max(), vmax=abs(importance).max())
plt.colorbar(label="restoration score (patching importance)")
plt.xlabel("head"); plt.ylabel("layer"); plt.title("Activation-patching importance per head")
plt.xticks(range(n_heads)); plt.yticks(range(n_layers))
plt.show()

# Top heads by importance
flat = [(importance[l, h], l, h) for l in range(n_layers) for h in range(n_heads)]
flat.sort(reverse=True)
print("Top 10 heads identified by activation patching:")
for score, l, h in flat[:10]:
    print(f"  layer {l:2d}, head {h:2d}   restoration = {score:.3f}")

You should see a **small number of heads light up**, concentrated in the middle-to-late layers, rather than importance spread everywhere. Those are the components activation patching says drive IOI. Reassuringly, they should overlap with the heads reported in the published IOI circuit (Wang et al., 2023) — that overlap is exactly the kind of ground-truth check we will quantify in a later stage.

## 5. The ablation harness
Now the second tool. To **ablate** a head we replace its output with its **average** output across the sentences ("mean ablation") — a neutral, information-free value. We then measure the logit difference. If switching a set of heads off makes the logit difference collapse, those heads were **necessary**.

This one function is what the faithfulness experiments will reuse.

In [ ]:
# Average head output across the clean sentences, kept per position (shape [1, pos, head, d_head])
mean_z = {l: clean_cache[utils.get_act_name('z', l)].mean(0, keepdim=True) for l in range(n_layers)}

def run_with_ablation(tokens, heads):
    """Mean-ablate the given (layer, head) pairs and return the logit difference."""
    by_layer = defaultdict(list)
    for l, h in heads:
        by_layer[l].append(h)

    def make_hook(layer, head_list):
        def hook(z, hook):
            for h in head_list:
                z[:, :, h, :] = mean_z[layer][:, :, h, :]
            return z
        return hook

    fwd_hooks = [(utils.get_act_name('z', l), make_hook(l, hs)) for l, hs in by_layer.items()]
    return logit_diff(model.run_with_hooks(tokens, fwd_hooks=fwd_hooks))

### Quick sanity check (a preview of faithfulness)
Ablating the **top patching heads** should make the logit difference drop sharply, while ablating the **same number of random heads** should barely matter. If that holds, the harness works and activation patching really did find the important components.

In [ ]:
top_heads = [(l, h) for _, l, h in flat[:8]]
all_heads = [(l, h) for l in range(n_layers) for h in range(n_heads)]
random.seed(1)
random_heads = random.sample(all_heads, 8)

print(f"Clean logit difference (nothing ablated):      {CLEAN_LD:+.3f}")
print(f"Ablating top 8 patching heads:                 {run_with_ablation(clean_tokens, top_heads):+.3f}   <- should collapse")
print(f"Ablating 8 random heads:                       {run_with_ablation(clean_tokens, random_heads):+.3f}   <- should stay high")

## What you've built
You now have the two instruments the whole study relies on:

- an **activation-patching importance score** for every head (the trusted baseline method), and
- a **mean-ablation harness** for testing whether a set of heads is genuinely necessary.

**Next — Stage 3:** get the *attribution / information-flow* importances (the LM Transparency Tool method). Then Stage 4 compares the two methods' head lists to each other and to the known circuit, and uses this ablation harness to measure how faithful each one is.

Save the `importance` matrix and commit this notebook to your repository.